# Etapa 4 — Avaliação

**Sumário do trabalho**

1. **Análise exploratória** (`01_analise_exploratoria.ipynb`): base, faltantes, alvo e análise bivariada.
2. **Tratamento dos dados** (`02_tratamento_dados.ipynb`): remoção de vazamento; faltantes (estrutural, exame não pedido, esporádico); codificação das qualitativas; remoção de colunas vazias, constantes e redundantes.
3. **Modelos** (`03_modelos.ipynb`): divisão treino/teste 80/20; árvore de decisão podada, regressão logística Lasso (seleciona os atributos) e Random Forest; hiperparâmetros por validação cruzada de 10 *folds*.
4. **Avaliação** (`04_avaliacao.ipynb`): AUC na validação cruzada e no teste, com intervalo de confiança e comparação com o acaso; matriz de confusão; uso clínico (limiar para não perder eventos e curva de decisão); interpretação dos três modelos. ← *este notebook*

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().resolve().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src import utils, preprocessing, features, train, evaluate

utils.setup_plotting()
pd.set_option("display.max_colwidth", 80)

### Dados e modelos

Mesmas funções e mesma semente da etapa 3: a divisão e os modelos ajustados são idênticos.

In [ ]:
patients, drugs, data_dictionary = preprocessing.load_data()
variable_groups = preprocessing.get_variable_groups(patients)
X, y, _ = features.build_dataset(patients, drugs, variable_groups)
X_train, X_test, y_train, y_test = preprocessing.split_train_test(X, y)

cv_results, searches = train.tune_models(X_train, y_train)
fitted = {name: search.best_estimator_ for name, search in searches.items()}

### Escolha do modelo

Pela validação cruzada do treino, sem olhar o teste: maior AUC e, entre os modelos a até um erro padrão dele, o mais simples.

In [ ]:
CHOSEN_MODEL = evaluate.choose_model(
    cv_results, simplest_first=["Árvore de decisão", "Regressão logística Lasso", "Random Forest"])

print(f"Modelo escolhido: {CHOSEN_MODEL}")
cv_results[["auc_cv", "auc_cv_erro_padrao"]]

### Teste: os modelos são melhores que o acaso?

No teste, 58,5% das internações não têm evento. Duas referências de acaso:
- **sempre a classe majoritária**: responde sempre "sem evento" e acerta 58,5% sem aprender nada;
- **sorteio estratificado**: chuta "evento" em 41,5% dos casos, ao acaso.

A **acurácia** se deixa enganar pela proporção das classes, por isso a comparação principal é pela **AUC** e pela **acurácia balanceada** (média de sensibilidade e especificidade), que valem 0,5 para o acaso qualquer que seja essa proporção. Intervalos de confiança de 95% por *bootstrap* (2000 reamostras do teste). `melhor_que_o_acaso`: os intervalos de AUC e de acurácia balanceada ficam inteiros acima de 0,5. `acuracia_acima_da_majoritaria`: o intervalo da acurácia fica inteiro acima de 58,5%.

In [ ]:
models = {**evaluate.chance_references(X_train, y_train), **fitted}
test_results = evaluate.test_metrics(models, X_test, y_test)

test_results.to_csv(utils.RESULTS_DIR / "model_test_results.csv")
test_results

Curva ROC no teste: quanto mais longe da diagonal (acaso), melhor.

In [ ]:
fig = evaluate.plot_roc_curves(fitted, X_test, y_test)
utils.save_fig(fig, "roc_curves_test")

### Matrizes de confusão no teste

Linhas: desfecho real; colunas: previsão.

In [ ]:
for name, model in fitted.items():
    print(name)
    display(evaluate.confusion_table(model, X_test, y_test))

### Uso clínico: limiar para não perder eventos

O modelo serviria para sinalizar pacientes para acompanhamento mais próximo; perder um evento (falso negativo) custa mais que um acompanhamento desnecessário. Comparação, no modelo escolhido, entre o limiar padrão (0,5) e um **limiar clínico**, escolhido no treino (validação cruzada) para identificar 90% dos eventos.

`sem_evento_entre_nao_sinalizados`: dos pacientes que o modelo deixaria fora do acompanhamento, quantos de fato não tiveram evento (ao acaso seriam 58,5%).

`limiar_curva_de_decisao`: o mesmo limiar na escala da curva de decisão abaixo (probabilidades corrigidas do efeito dos pesos por classe).

In [ ]:
chosen = fitted[CHOSEN_MODEL]
CLINICAL_THRESHOLD = evaluate.clinical_threshold(chosen, X_train, y_train, target_sensitivity=0.90)

thresholds = {"Limiar padrão (0,5)": 0.5, "Limiar clínico (90% dos eventos no treino)": CLINICAL_THRESHOLD}
threshold_results = evaluate.threshold_comparison(chosen, X_test, y_test, thresholds,
                                                 train_prevalence=y_train.mean())
threshold_results.to_csv(utils.RESULTS_DIR / "clinical_threshold_results.csv", index_label="estrategia")
threshold_results

Matrizes de confusão do modelo escolhido com os dois limiares.

In [ ]:
fig = evaluate.plot_confusion_matrices(chosen, X_test, y_test, {
    "Limiar padrão (0,5)": 0.5,
    f"Limiar clínico ({CLINICAL_THRESHOLD:.3f})".replace(".", ","): CLINICAL_THRESHOLD,
})
utils.save_fig(fig, "confusion_matrices_lasso")

**Curva de decisão** (Vickers e Elkin, 2006): benefício líquido de cada estratégia para cada limiar de probabilidade. O limiar exprime quanto pior é perder um evento: 0,25 significa que um evento perdido vale 3 acompanhamentos desnecessários; 0,10, que vale 9. Um modelo só é útil onde sua curva fica acima de "sinalizar todos" e de "não sinalizar ninguém". As probabilidades são corrigidas do efeito dos pesos por classe.

In [ ]:
curves = evaluate.decision_curve(fitted, X_test, y_test, train_prevalence=y_train.mean())
curves.to_csv(utils.RESULTS_DIR / "decision_curve.csv")

fig = evaluate.plot_decision_curve(curves)
utils.save_fig(fig, "decision_curve")

### Interpretação

**Árvore de decisão.** Cada caixa mostra a regra, a proporção das internações do treino que chega ali e a proporção de cada classe (ponderada pelos pesos por classe).

In [ ]:
fig = evaluate.plot_tree(fitted["Árvore de decisão"], X_train.columns)
utils.save_fig(fig, "decision_tree")

**Lasso: os 15 maiores coeficientes.** Razão de chances por desvio padrão: acima de 1 aumenta o risco de óbito ou readmissão, abaixo de 1 reduz.

In [ ]:
lasso_table = evaluate.lasso_coefficients(fitted["Regressão logística Lasso"], X_train.columns)
lasso_table.to_csv(utils.RESULTS_DIR / "lasso_coefficients.csv", index_label="atributo")
lasso_table

**Random Forest: os 15 atributos mais importantes.** Queda da AUC no teste quando o atributo é embaralhado.

In [ ]:
forest_table = evaluate.forest_importance(fitted["Random Forest"], X_test, y_test)
forest_table.to_csv(utils.RESULTS_DIR / "forest_importance.csv", index_label="atributo")
forest_table

### Conclusões

- **Os três modelos são melhores que o acaso**: AUC e acurácia balanceada com intervalo de confiança inteiro acima de 0,5.
- **A acurácia, sozinha, não mostra esse ganho.** Lasso e Random Forest acertam cerca de 60%, pouco acima dos 58,5% de responder sempre "sem evento", e o intervalo de confiança inclui esse valor. A árvore acerta menos que isso, embora identifique a maioria dos eventos. Por isso a avaliação se apoia na AUC e na acurácia balanceada.
- **Lasso e Random Forest empatam** na validação cruzada e no teste; fica o **Lasso**, mais simples e interpretável. A árvore única fica abaixo dos dois, o que mostra o ganho do *bagging* no Random Forest.
- **O limiar muda o uso do modelo, não a sua qualidade.** Com o limiar padrão o Lasso identifica cerca de metade dos eventos; com o limiar clínico, cerca de 84%, ao custo de sinalizar três quartos dos pacientes. Os pacientes que ele deixa de fora têm 75% de chance de não ter evento, contra 58,5% ao acaso.
- **A curva de decisão limita o argumento clínico.** Quando perder um evento vale mais que cerca de quatro acompanhamentos desnecessários (limiar abaixo de 0,2), os modelos não superam sinalizar todos (Lasso e Random Forest praticamente empatam; a árvore fica abaixo). A partir de cerca de 0,25, o Lasso supera as duas estratégias padrão até cerca de 0,5. Na escala da curva, o limiar padrão vale 0,415 e o clínico 0,279: os dois caem na faixa em que o modelo ajuda.
- **O ganho existe, mas é modesto** (AUC de cerca de 0,63 no teste): com os dados da internação, a previsão de óbito ou readmissão em 6 meses é limitada.
- **Os modelos concordam nos atributos principais**: tipo de insuficiência cardíaca, diâmetro do ventrículo esquerdo, medidas do ecocardiograma não registradas e função renal.
- **Os indicadores de exame não pedido ajudam**: sem eles, a AUC do Lasso na validação cruzada cai de 0,673 para 0,647 (etapa 3).